In [1]:
import math
import torch
from torch import nn
from torch import Tensor
from torch.nn  import functional as F
import gpytorch
from matplotlib import pyplot as plt
from torch.distributions.multivariate_normal import MultivariateNormal
import matplotlib.cm as cm
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D 
import sys
from decimal import Decimal
from IPython.display import clear_output
sys.path.append("..")
from LBFGS import FullBatchLBFGS
from kernels import vvkernels as vvk, sep_vvkernels as svvk, vvk_rbfkernel as vvk_rbf
from means import vvmeans as vvm
from likelihood import vvlikelihood as vvll
from mlikelihoods import MarginalLogLikelihood as exmll
from predstrategies import GPprediction
from utils import ObjFun, get_vertices, stopping_criteria
from scipy import stats
import numpy as np
import seaborn as sns
import scipy
from ALDmodel import ALDGrowth
from physicsmodel import ALDOpt
from core import plot_2d, plot_uq, WP
from functools import partial
from scipy.stats import halfcauchy, triang
import botorch
from botorch.acquisition import ExpectedImprovement
from botorch.models import SingleTaskGP
import warnings
# torch.manual_seed(0)
# np.random.seed(0)
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
sns.set_style('whitegrid') # darkgrid, white grid, dark, white and ticks
plt.rc('axes', titlesize=34)     # fontsize of the axes title
plt.rc('axes', labelsize=34)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=34)    # fontsize of the tick labels
plt.rc('ytick', labelsize=34)    # fontsize of the tick labels
plt.rc('legend', fontsize=30)    # legend fontsize
plt.rc('font', size=26)     

In [3]:
torch.set_default_dtype(torch.float64)

In [4]:
#############################################################
def ALDsample(x, bnds=None, apf=None, output=True,
              nmult=0.5, nrep=5, stdgrowth=None,
              noise=0.1, wrtfile='log.txt'):

    x = 10**np.array(x)

    bm = 10**bnds[:, 1]
    tmax = np.sum(bm)

    st = apf.cycle(bm[0], bm[1], bm[2], bm[3], 1)  # throw-away cycle

    if stdgrowth is None:
        stdgrowth = st[0]

    xp = x + np.ones((4,))
    g = apf.cycle(x[0], x[1], x[2], x[3], nrep + 1)
    gp = apf.cycle(xp[0], xp[1], xp[2], xp[3], nrep + 1)
    
    gmaxL = [st[0]]
    g0L = [g[0]]
    for ii in range(nrep - 1):
        gmaxL += [apf.cycle(bm[0], bm[1], bm[2], bm[3], 1)[0]]
        g0L += [apf.cycle(x[0], x[1], x[2], x[3], 1)[0]]
    gmax = np.mean(gmaxL)
    g0 = np.mean(g0L)

    gm = np.mean(g[1:])
    gpm = np.mean(gp[1:])

    if output:
        WP('x: ' + str(x), wrtfile)
        WP('g: ' + str(np.round(g, 3)), wrtfile)
        WP('gp: ' + str(np.round(gp, 3)), wrtfile)
        WP('g0L:' + str(np.round(g0L, 3)), wrtfile)

    vc1 = (np.abs(gm - gpm)/noise) - nmult
    vc2 = (np.abs(gm - g0)/noise) - nmult

    if vc1 < nmult:
        vc1 = nmult
    if vc2 < nmult:
        vc2 = nmult

    gc = (stdgrowth - gm)**2
    tc = np.sum(x)/tmax

    y = gc + vc1 + vc2 + tc

    if output:
        WP('g, gc, vc1, vc2, tc, y: ' + \
           str(np.round(g[-1], 4)) + ' ' + \
           str(np.round(gc, 4)) + ' ' + \
           str(np.round(vc1, 4)) + ' ' + \
           str(np.round(vc2, 4)) + ' ' + \
           str(np.round(tc, 4)) + ' ' + \
           str(np.round(y, 4)), wrtfile)

    return np.log10(y).reshape(1,) , g[-1].reshape(1,), gc.reshape(1,1), vc1*np.ones(1,), vc2*np.ones(1,), tc

######ALD HELPERS############
def ALDinitialize(system='Al2O3-200C', noise=0.01):

    # chem = (p, M, beta, tp)
    # chem is the tuple of chemical parameters
    # p: precursor pressure (Pa)
    # M: molecular mass (atomic mass units)
    # beta: sticking probability
    # tp: characteristic time of precursor evacuation
    if system == 'Al2O3-200C':
        chem1 = (26.66, 72, 1e-3, .2, 1.0)
        chem2 = (26.66, 18, 1e-4, .2, 0.0)
        T = 473  # temperature in K
        sitearea = 0.225e-18  # area of a surface site, in m^2

    elif system == 'Al2O3-100C':
        chem1 = (26.66, 72, 1e-4, 3, 1.0)
        chem2 = (26.66, 18, 1e-5, 10, 0.0)
        T = 373  # temperature in K
        sitearea = 0.251e-18  # area of a surface site, in m^2

    elif system == 'TiO2-200C':
        chem1 = (0.6665, 284, 1e-4, .2, 1.0)
        chem2 = (26.66, 18, 1e-4, .2, 0.0)
        T = 473  # temperature in K
        sitearea = 1.17e-18  # area of a surface site, in m^2

    elif system == 'W-200C':
        chem1 = (6.665, 297, 0.2, .2, 1.0)
        chem2 = (26.66, 62, 0.05, .2, 0.0)
        T = 473  # temperature in K
        sitearea = 0.036e-18  # area of a surface site, in m^2

    apf = ALDGrowth(T, sitearea, chem1, chem2, noise)

    return apf


#################################
def lhs_design(npt, ndim, lwr, upr, output=True, wrtfile='log.txt'):
    fname = 'maximin_lhs_l2_' + str(npt) + '_' + str(ndim) + 'd.csv'
    stset = np.loadtxt(fname, delimiter=',')
    stset = (np.log10(upr)-np.log10(lwr))*(stset/np.int16(npt-1)) + np.log10(lwr)
    if output:
        WP(str(10**stset), wrtfile)
    return stset
############################
wrtfile='log.txt'

ninit = 10
niter = 40
nopt = 10 #10
ndim = 4
nmult = 1.
#nrepL = [8] #[5, 10, 20, 40]
output = False
plot_freq = 100
acq = 'EI'
stpt = 1.0
model_type = 'GP'
loocv = False
lwr = 0.2
system = 'Al2O3-200C'
noise_imposed = 0.1 #1   #'#1 #0.1 #0.1 #0.001
##############################
apf_ = ALDinitialize(system, noise_imposed)
func = partial(ALDsample, apf=apf_)
optval = None

"""for ALD0"""
uprL = [4] #[.5, 1, 2, 4, 8, 16, 32, 64, 128, 256]
for upr in uprL:
    print(upr)
    bnds_ = upr*np.ones((ndim, 2)).astype('float')
    bnds_[:, 0] = lwr
    bnds_ = np.log10(bnds_)
    print(bnds_)
    bm = 10**bnds_[:, 1]
    st = apf_.cycle(bm[0], bm[1], bm[2], bm[3], 100)
    noise = np.std(st[5:])
    stdgrowth = np.mean(st[5:])
    WP('bound: ' + str(upr) + ', noise: ' + str(noise), wrtfile)

    res = func(bnds_[:, 1], bnds_, output=True,
               nmult=nmult, stdgrowth=stdgrowth, noise=noise,
               wrtfile=wrtfile)
    print(res)
    if res[3] == nmult and res[4] == nmult:
        print('kjhg')
        WP('final upper bound: ' + str(upr), wrtfile)
        break
    
print(bnds_)
"""load the LHS design for ninit points"""
stset = lhs_design(npt=ninit, ndim=ndim,
                   lwr=lwr, upr=upr, wrtfile=wrtfile)

4
[[-0.69897     0.60205999]
 [-0.69897     0.60205999]
 [-0.69897     0.60205999]
 [-0.69897     0.60205999]]
bound: 4, noise: 0.09972218524367875
x: [4. 4. 4. 4.]
g: [0.943 1.021 1.009 1.068 0.962 1.118]
gp: [0.991 1.236 0.918 0.953 0.945 1.003]
g0L:[0.943 0.807 1.106 0.931 1.018]
g, gc, vc1, vc2, tc, y: 1.1175 0.0003 1.0 1.0 1.0 3.0003
(array([0.47716212]), array([1.11753545]), array([[0.00028228]]), array([1.]), array([1.]), 1.0)
kjhg
final upper bound: 4
[[-0.69897     0.60205999]
 [-0.69897     0.60205999]
 [-0.69897     0.60205999]
 [-0.69897     0.60205999]]
[[0.2        0.7572958  0.7572958  4.        ]
 [0.27899016 1.4736126  0.27899016 0.54288352]
 [0.38917754 0.38917754 2.05561707 0.38917754]
 [0.54288352 2.86748466 2.86748466 1.4736126 ]
 [0.7572958  0.2        0.2        1.05639038]
 [1.05639038 4.         1.05639038 0.2       ]
 [1.4736126  0.27899016 1.4736126  2.86748466]
 [2.05561707 2.05561707 0.38917754 2.05561707]
 [2.86748466 0.54288352 0.54288352 0.27899016]
 [4.

In [5]:
print(noise)

0.09972218524367875


In [6]:
print(10**bnds_)
print(bnds_.shape)

[[0.2 4. ]
 [0.2 4. ]
 [0.2 4. ]
 [0.2 4. ]]
(4, 2)


In [7]:

sample_size = 1
D = 4 #vf.D
N = 2 #3#vf.N

#




def vfield_(x,noise_value,apf_):
    
    x = x.reshape(x.shape[0],D)
    out = torch.zeros(x.shape[0], 1)
    gA = torch.zeros(x.shape[0], 1)
    vc1_c = torch.zeros(x.shape[0], 1)
    vc2_c = torch.zeros(x.shape[0], 1)

    for i in range(x.shape[0]):
        c_i, g_last, gc, vc1, vc2, tc= ( ALDsample(x[i].reshape(D,).numpy(),bnds=bnds_, apf=apf_, output=False,
              nmult=1., nrep=5, stdgrowth=None,
              noise=noise_value, wrtfile='log.txt')) 
        out[i] = Tensor(c_i) #(Tensor(gc)) #Tensor(c_i) #Tensor(gc) #Tensor(c_i) #vf(x[:,0], x[:,1]) + torch.randn(Tensor(vf(x[:,0], x[:,1])).size()) * math.sqrt(noise_value)
        gA[i] = (Tensor(g_last))
        vc1_c[i] = torch.log10(Tensor(vc1))
        vc2_c[i] = torch.log10(Tensor(vc2))
    #print(c_i)
    y =  torch.cat([out,gA]).reshape(N, x.shape[0]) #torch.cat([out,gA,Tensor(gc)]).reshape(N,1) #torch.cat([out,gA, vc1_c, vc2_c]).reshape(N,1) #torch.cat([out,gA]).reshape(N,1) #
    return y #out.reshape(1,1), gA.reshape(1,1) #y #Tensor(out), gA #/torch.max(out)




In [8]:
# x_train = train_x #loc #torch.linspace(0, 1, 10)
# y_train = train_y #v  #torch.stack([torch.sin(train_x * (2 * math.pi)) + torch.randn(train_x.size()) * 0.2,torch.cos(train_x * (2 * math.pi)) + torch.randn(train_x.size()) * 0.2,], -1)

class GPModel(gpytorch.models.ExactGP):
    
    def __init__(self, train_x, train_y, likelihood):
        super(GPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()  #vvm.TensorProductSubMean(gpytorch.means.LinearMean(2), num_tasks = 2)#vvm.TensorProductSubMean(gpytorch.means.ConstantMean(), num_tasks = 2)  # 

            
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.MaternKernel(nu = 2.5, ard_num_dims = train_x.shape[1]))#gpytorch.kernels.ScaleKernel(gpytorch.kernels.PolynomialKernel(6)) #

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)
    @property 
    def num_outputs(self): 
        #For a single-task GP, this is typically 1 
        return 1

In [9]:
num_base_kernels = 2
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood,num_base_kernels):
        super(MultitaskGPModel, self).__init__(train_x, train_y, likelihood)
  
        self.mean_module = vvm.TensorProductSubMean(gpytorch.means.ZeroMean(), num_tasks = N)  #vvm.TensorProductSubMean(gpytorch.means.LinearMean(2), num_tasks = 2)#vvm.TensorProductSubMean(gpytorch.means.ConstantMean(), num_tasks = 2)  # 
        base_kernels = []
        for i in range(num_base_kernels):
            base_kernels.append(gpytorch.kernels.ScaleKernel(( gpytorch.kernels.MaternKernel(nu = 2.5,ard_num_dims = train_x.shape[1]) ))) #gpytorch.kernels.PolynomialKernel(4)  ##gpytorch.kernels.MaternKernel()# (vvk_rbf.vvkRBFKernel())
 
            
        self.covar_module = svvk.SepTensorProductKernel(base_kernels,num_tasks = N)

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

In [10]:
# # ###hyperparameters optimization###
def gp_fit(g_theta1, agg_data, training_iter, current_model = None, current_likelihood = None):
   # noises = torch.ones(agg_data.shape[0]) * (noise_value) #  torch.zeros(agg_data.shape[0]) # 
   # noises = noises.reshape(g_theta1.shape[0], 2)
    
#     if (current_model is not None):
#         likelihood = current_likelihood #vvll.FixedNoiseMultitaskGaussianLikelihood(2, noises) #vvll.FixedNoiseMultitaskGaussianLikelihood(2, noises)  #

#         model = current_model#.get_fantasy_model(g_theta1, agg_data) #MultitaskGPModel(g_theta1, agg_data, likelihood,num_base_kernels)
#         model.set_train_data(g_theta1, agg_data,  strict=False)
#     else:
#         likelihood = vvll.FixedNoiseMultitaskGaussianLikelihood(noises) #vvll.TensorProductLikelihood(num_tasks = 2)#vvll.FixedNoiseMultitaskGaussianLikelihood(2, noises) #
#         model = MultitaskGPModel(g_theta1, agg_data, likelihood,num_base_kernels)
        
     #vvll.TensorProductLikelihood(num_tasks = 2) #
    likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=N)
   # model = GPModel(g_theta1, agg_data, likelihood)
    model = MultitaskGPModel(g_theta1, agg_data, likelihood,num_base_kernels)
    
    model.double()
    likelihood.double()


    model.train()
    
    likelihood.train()
    

    optimizer = torch.optim.Adam(model.parameters(),  lr= .07) #, weight_decay=0.001)  # Includes GaussianLikelihood parameters
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model) #
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
    

    for i in range(training_iter):
        optimizer.zero_grad()
        output = model(g_theta1)
    # Calc loss and backprop gradients
        loss = - mll(output, agg_data)
       
       # loss, chi_square  = mll(agg_data,g_theta1, model, likelihood, noise_value) #
#         loss = -1. * loss

        loss.backward()
        optimizer.step()
       # scheduler.step(loss)



    
        
    print('loss is %.3f' %loss)
#     for params in model.named_parameters():
#         print(params)
    return model, likelihood

In [17]:
### design search
def conduct_design_search(x0,loc_sample, f_target, g_theta1, agg_data, model, likelihood,lkl, training_design_iter, lr_new,h,box,ei):
    noise_value =noise**2 #llkl.noise.detach()# np.std(st[5:]) #lkl.noise.detach()#
    g_theta2_center = nn.Parameter(Tensor(loc_sample))

    x_d= nn.Parameter(Tensor(x0))
    
    optimizer = torch.optim.Adam([{'params': g_theta2_center, 'lr': 0.0001},{'params': x_d, 'lr': 0.0001}])

    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
    loss2_old = 0.0
    count = 0
    for ii in range( training_design_iter ):
#         x_d = torch.cat([x_d_0, x_d_1]).reshape(1,2)
#         g_theta2 = torch.cat([g_theta20, g_theta21],1)
        optimizer.zero_grad()
        #loss3 = ei(x_d)
        loss2, pf1, Qf1, Qf12, data_fit, Q21 = likelihood.get_ell_(agg_data,f_target,x_d, g_theta1, model, noise_value, g_theta2_center,h,box,N)
        #print(x_d)
       # print(loss3)
        loss2 = -1. * loss2 #-1. * loss2 + loss3 #loss3 #-1. * loss2 #+ loss3 #loss3 #
       # print(loss2)
        loss2.backward()
        if torch.abs(loss2 - loss2_old) < 1e-4:
            count += 1
            if count >= 10:
                break;
        else:
            count = 0
        loss2_old = loss2.detach()
        optimizer.step()
        
#     loss2, pf1, Qf1, Qf12, data_fit, Q21 = likelihood.get_bopt_ell(agg_data,f_target,x_d, g_theta1, model, likelihood, noise_value, g_theta2, theta)
#     loss2 = -1. * loss2
    print('Loss design: %.3f' % ( loss2))
#     #print(x_d)
    return x_d, g_theta2_center, loss2, pf1, Qf1, Qf12, data_fit, Q21

In [18]:
# x0 = Tensor(lhs_design(npt=10, ndim=ndim,
#                    lwr=lwr, upr=upr, wrtfile=wrtfile))[0]

In [19]:
iter_max = 40
x0_tot = dict()
gA_tot = dict()

bestA_tot = dict()
noise_value =noise_imposed**2
sampling_2 = True
x0_ini = Tensor([0.0, 0.0, 0.0, 0.0]).reshape(1,D)#-0.1 * torch.ones(1,D) # torch.rand(1,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0] #Tensor(1.*(bnds_[:, 1])) #
for ii in range(nopt):
    train_x = Tensor(stset)[0:sample_size].reshape(sample_size,D)
    apf_ = ALDinitialize(system, noise_imposed)
    train_y = vfield_(train_x, noise, apf_)
    print(train_y)
    print('num iter')
    print(ii)

    #x0 = Tensor(1.*(10**bnds_[:, 1])) #torch.rand(1,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0] #Tensor(1.*(10**bnds_[:, 1]))
    #x0 = x0.reshape(1,D)
    #print(x0)
    iter_hp = 75
    iter_design = 2000
    g_theta1 = train_x
    agg_data = train_y.flatten()

    f_target = Tensor([0.4, 1.]).reshape(N,1) #0.4 * torch.ones(1,1)#Tensor([[0.0, 1., np.log10(1.), np.log10(1.)]]).reshape(N,1)# 0.4 * torch.ones(1,1)#0.0 + torch.ones(2,1)
    tol_vector = 0.1 * torch.ones(f_target.shape)
   # tol_vector_y = 0.05 * torch.ones(f_target.shape)
    ubnds = np.zeros(bnds_.shape)
    ubnds[:, 1] = 4.
    box = bnds_ #np.round(10**np.array([bnds_])
    ###2 points ini
    N_2 = 1
    loc_size = N_2
    loc_sample = dis_2sample = MultivariateNormal( loc = x0_ini, covariance_matrix= .01 * torch.eye(D) )
                    #loc_size = 4
    loc_sample = dis_2sample.sample((N_2,)).reshape(N_2,D) #torch.rand(N_2,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0]
    g_theta2_vec = (Tensor(loc_sample).clone()).flatten()
    
    ###Target design ini

   # best_x, gA_vec = vfield_(x0.clone(), np.std(st[5:]))
    best_x =  train_y[0,0].reshape(1,1) #best_x.reshape(1,1)
    gA_vec = train_y[1,0].reshape(1,1)
    gA2_vec = torch.empty(1,1)
    
    SUCCESS = False 
    FAILURE = False 
    lr_new = 1.
    iter = 1
    h = 0.001
    g_theta2_vec = (Tensor(loc_sample).clone()).flatten()
    
    data_fit_vec = torch.empty((1,1))
    entropy_vec = torch.empty((1,1))

    while(SUCCESS == False and FAILURE == False):
        print(iter)
        print('START HYPERPARAMETERS optimization')

        model, lkl = gp_fit(g_theta1,agg_data,iter_hp)
        noise_value =noise**2
        noises = torch.ones(agg_data.shape[0]) * (noise_value) #  torch.zeros(agg_data.shape[0]) # 
        noises = noises.reshape(g_theta1.shape[0], N)

        likelihood_tad =  vvll.FixedNoiseMultitaskGaussianLikelihood(noises) #TadFixedNoiseGaussianLikelihood(np.std(st[5:]))
        print('END HYPERPARAMETERS optimization')
        if (iter < ninit - sample_size + 1):
            x0_new = Tensor(stset[iter, :].reshape(1,D))  
            new_data_x = vfield_(x0_new.detach(),noise_imposed,apf_)
            agg_data = torch.cat([agg_data, new_data_x.flatten()], 0)
            g_theta1= torch.cat([g_theta1, x0_new.detach()], 0)
            best_x_current = new_data_x[0,0].reshape(1,1)
            gA_current = new_data_x[1,0].reshape(1,1)
            best_x = torch.cat([best_x, best_x_current], 0)
            gA_vec = torch.cat([gA_vec, gA_current], 0)
            iter = iter+1
            #x0 = #x0_new.detach()
            x0 = x0_ini#Tensor((bnds_[:, 1])) #torch.rand(1,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0] #Tensor(1.*(10**bnds_[:, 1]))
            x0 = x0.reshape(1,D)
           # print(x0)
            #g_theta2 = torch.rand(N_2,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0]
        
        else:
            loc_sample_old = loc_sample.clone()
            ei = None #ExpectedImprovement(model = BotorchWrappedGP(model), best_f = best_f, maximize = False)
            model.eval()
            lkl.eval()
            x0_new,g_theta2, loss, pf1, Qf1, Qf12, data_fit, Q21 = conduct_design_search(x0, loc_sample, f_target, g_theta1, agg_data, model, likelihood_tad,lkl, iter_design, lr_new,h,box,ei)
        # x0_new = torch.zeros(x0_new_.shape)
        # print(x0_new.shape)
        # for ii in range(bnds_.shape[0]):
        #     x0_new[0,ii] = x0_new_[0,ii]*(bnds_[ii, 1] - bnds_[ii, 0]) + bnds_[ii, 0]
        
        # g_theta2 = torch.zeros(g_theta2_.shape)
        # for ii in range(bnds_.shape[0]):
        #     g_theta2[:,ii] = g_theta2_[:,ii]*(bnds_[ii, 1] - bnds_[ii, 0]) + bnds_[ii, 0]
            cur_model = model
            cur_likelihood = lkl
        
      
            lower_bound = torch.zeros(pf1.shape)
            upper_bound = torch.zeros(pf1.shape)
    
            for i in range(pf1.shape[0]):
                lower_bound[i] = pf1[i] -  2. * torch.sqrt(Qf12[i,i]) # -  2. * torch.sqrt(Qf12[i,i])
                upper_bound[i] = pf1[i]+  2. * torch.sqrt(Qf12[i,i])# +  2. * torch.sqrt(Qf12[i,i])

            print(pf1)
            print(lower_bound)
            print(upper_bound)
        
            SUCCESS = stopping_criteria(tol_vector, pf1.detach(), f_target, lower_bound, upper_bound)
            print(SUCCESS)
        
            entropy = 0.
            
            print('expected info is '+str(entropy))
            iter = iter+1
            if (iter >= iter_max):# (torch.abs(entropy) < 0.0 * 1e-4 * tol_vector[0,0]):
                FAILURE = True
            else:
        #print('mohabb disatance is' + str(Qf12.inv_quad(f_target - pf1)))
                if not SUCCESS:
            
            
                    with torch.no_grad():
    
                
                        if iter >= 0:
                            new_data_x= vfield_(x0_new.detach(),noise_imposed,apf_) #lkl.noise.detach())# 
                          
                            if sampling_2 == True:
                                new_data_g2 = vfield_(g_theta2.detach(),noise_imposed,apf_)#lkl.noise.detach())# 
                                print('current sol is'+str(x0_new.detach()))
                                best_x_current = new_data_x[0,0].reshape(1,1)
                                gA_current = new_data_x[1,0].reshape(1,1)
                                best_x = torch.cat([best_x, best_x_current], 0)
                                gA_vec = torch.cat([gA_vec, gA_current], 0)
                                best_x_2 = new_data_g2[0,0].reshape(1,1)
                                gA_2 = new_data_g2[1,0].reshape(1,1)
                                gA2_vec = torch.cat([gA2_vec, gA_2], 0)
                                best_x = torch.cat([best_x, best_x_2], 0)
                                gA_vec = torch.cat([gA_vec, gA_2], 0)
                                              #  gA_vec = torch.cat([gA_vec, gA], 0)
                            # iter = iter + 1
                                agg_data = torch.cat([agg_data, new_data_g2.flatten()], 0)
                             
                                g_theta1= torch.cat([g_theta1, g_theta2.detach()], 0)
                                x0 = x0_new.detach() #Tensor(1.*(bnds_[:, 1])).reshape(1,D) #torch.ones(1,D)# x0_ini.detach() #x0_new.detach()
                                dim = x0_new.detach().shape[1]
                                for i in range(dim):
                                    if x0[0,i] < box[i,0]:
                                        x0[0,i] = box[i,0]
                                    elif x0[0,i] > box[i,1]:
                                        x0[0,i] = box[i,1]

                                sum = torch.zeros(D, D)#.to(device)
                                mean_2 = torch.mean(g_theta2.detach(), 0, True)
            
                                for i in range(N_2):
                                    #sum =sum + torch.matmul((g_theta2.detach()[i] -mean_2).t(), ( g_theta2.detach()[i] - mean_2 ) )# sum + torch.matmul((g_theta2.detach()[i] -x0_new.detach()).t(), (g_theta2.detach()[i] - x0_new.detach()) ) #sum + torch.matmul((g_theta2.detach()[i] -x0_new.detach()).t(), (g_theta2.detach()[i] - x0_new.detach()) ) # 
                                    sum =sum + torch.matmul((g_theta2.detach()[i] -x0_new.detach()).t(), (g_theta2.detach()[i] - x0_new.detach()) ) #sum + torch.matmul((g_theta2.detach()[i] -
                                emp_cov = 1./N_2 * sum + torch.eye(sum.shape[0]) * 1e-8
            
                                dis_2sample = MultivariateNormal( loc = x0_new.detach(), covariance_matrix=emp_cov )
                                #loc_size = 4
                                loc_sample = dis_2sample.sample((N_2 - 1,))
            
                                loc_sample = loc_sample.reshape(N_2 - 1, D)
                                loc_sample = torch.cat([loc_sample, x0_new.detach()],0) #torch.rand(N_2,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0] #torch.cat([loc_sample, x0_new.detach()],0) #g_theta2.detach() #torch.cat([loc_sample, x0_new.detach()],0)
                                dim = loc_sample.shape[1]
                                for i in range(dim):
                                    for j in range(N_2):
                                        if loc_sample[j,i] < box[i,0]:
                                            loc_sample[j,i] = box[i,0]
                                        elif loc_sample[j,i] > box[i,1]:
                                            loc_sample[j,i] = box[i,1]
                            
                            #####YOU NEED TO UPDATE THIS to reflect start of 2
                            else:
                                loc_sample = g_theta2.detach()
                                print('current sol is'+str(x0_new.detach()))
                                best_x_current = new_data_x[0,0].reshape(1,1)
                                gA_current = new_data_x[1,0].reshape(1,1)
                                best_x = torch.cat([best_x, best_x_current], 0)
                                gA_vec = torch.cat([gA_vec, gA_current], 0)
                                              #  gA_vec = torch.cat([gA_vec, gA], 0)
                            # iter = iter + 1
                                agg_data = torch.cat([agg_data, new_data_x.flatten()], 0)
                             
                                g_theta1= torch.cat([g_theta1, x0_new.detach()], 0)
                                x0 = x0_new.detach() #Tensor(1.*(bnds_[:, 1])).reshape(1,D) #torch.ones(1,D)# x0_ini.detach() #x0_new.detach()
                                dim = x0_new.detach().shape[1]
                                for i in range(dim):
                                    if x0[0,i] < box[i,0]:
                                        x0[0,i] = box[i,0]
                                    elif x0[0,i] > box[i,1]:
                                        x0[0,i] = box[i,1]
                                
                            #torch.rand(N_2,D) * (bnds_.T[1] - bnds_.T[0]) + bnds_.T[0] #Tensor(vf.high_1  * np.random.random_sample((loc_size + 1,2)) + vf.low_1)
                                               
      
               
            print('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
            
    

    new_data_x= vfield_(x0_new.detach(),noise,apf_) #lkl.noise.detach())# 

    print('current sol is'+str(x0_new.detach()))
    best_x_current = new_data_x[0,0].reshape(1,1)
    gA_current = new_data_x[1,0].reshape(1,1)
    best_x = torch.cat([best_x, best_x_current], 0)
    gA_vec = torch.cat([gA_vec, gA_current], 0)
   # gA_vec = torch.cat([gA_vec, gA], 0)
# iter = iter + 1
    agg_data = torch.cat([agg_data, new_data_x.flatten()], 0)
    best_f = agg_data.min()
    g_theta1= torch.cat([g_theta1, x0_new.detach()], 0)
    
    x0_tot[str(ii)] = x0_new.detach()
    bestA_tot[str(ii)] = dict()
    bestA_tot[str(ii)] ['cost'] = best_x
    bestA_tot[str(ii)]['iter'] = iter
    gA_tot[str(ii)] = dict()
    gA_tot[str(ii)]['gA'] = gA_vec
    gA_tot[str(ii)]['iter'] = iter
    gA_tot[str(ii)]['gA2'] = gA2_vec
    if sampling_2:
        torch.save(bestA_tot, '../data_plots/sample2/best_A_tot_'+str(noise_imposed))
        torch.save(gA_tot, '../data_plots/sample2/gA_tot_'+str(noise_imposed))
        torch.save(x0_tot, '../data_plots/sample2/x0_tot_'+str(noise_imposed))
    
   # x0_tot['ii']['times'] = x0_new.detach()
    
    print('current sol is'+str(x0_new.detach()))
    # best_x = torch.cat([best_x, new_data_x], 0)
    # gA_vec = torch.cat([gA_vec, gA], 0)
        
    print('Success is ' + str(SUCCESS) + ' and failure is ' + str(FAILURE)+' after '+ str(iter) + ' iterations')
       
    # torch.save(best_A_tot, 'data/best_A_tot')
    

tensor([[0.3825],
        [1.0241]])
num iter
0
1
START HYPERPARAMETERS optimization
loss is 0.082
END HYPERPARAMETERS optimization
2
START HYPERPARAMETERS optimization
loss is -0.303
END HYPERPARAMETERS optimization
3
START HYPERPARAMETERS optimization
loss is 0.489
END HYPERPARAMETERS optimization
4
START HYPERPARAMETERS optimization
loss is 0.383
END HYPERPARAMETERS optimization
5
START HYPERPARAMETERS optimization
loss is 0.415
END HYPERPARAMETERS optimization
6
START HYPERPARAMETERS optimization
loss is 0.528
END HYPERPARAMETERS optimization
7
START HYPERPARAMETERS optimization
loss is 0.543
END HYPERPARAMETERS optimization
8
START HYPERPARAMETERS optimization
loss is 0.528
END HYPERPARAMETERS optimization
9
START HYPERPARAMETERS optimization
loss is 0.441
END HYPERPARAMETERS optimization
10
START HYPERPARAMETERS optimization
loss is 0.398
END HYPERPARAMETERS optimization
Loss design: -3.304
tensor([[0.4194],
        [1.0625]], grad_fn=<ViewBackward0>)
tensor([[0.1646],
        [0

In [20]:
if sampling_2:
    torch.save(bestA_tot, '../data_plots/sample2/best_A_tot_'+str(noise_imposed))
    torch.save(gA_tot, '../data_plots/sample2/gA_tot_'+str(noise_imposed))
    torch.save(x0_tot, '../data_plots/sample2/x0_tot_'+str(noise_imposed))

In [ ]:
new_data_x = vfield_(x0_new.detach(),noise, apf_)

In [ ]:
print(new_data_x)

In [ ]:
cm = sns.color_palette("husl", 8)
cycL_random = torch.arange(0, 830, 21)
best_random = torch.tensor([[1.2582613652870023, 1.1780128028162413, 1.0198339754678056, 1.0198339754678056, 1.0198339754678056, 1.0198339754678056, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802, 0.4001391922678802], [0.7132201635568373, 0.7132201635568373, 0.7132201635568373, 0.7132201635568373, 0.7132201635568373, 0.7132201635568373, 0.7132201635568373, 0.6399003078264976, 0.6399003078264976, 0.44640207926538666, 0.44640207926538666, 0.44640207926538666, 0.44640207926538666, 0.44640207926538666, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.43530565740766447, 0.4185681069190185, 0.4185681069190185, 0.4185681069190185, 0.4185681069190185], [1.1565686129428911, 1.1565686129428911, 1.1565686129428911, 0.8929410237997912, 0.8929410237997912, 0.8929410237997912, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.4174374201017401, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135, 0.3966387126936135], [0.9080379076846883, 0.9080379076846883, 0.9080379076846883, 0.9080379076846883, 0.9080379076846883, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.43898490545157254, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653, 0.4024692279280653], [1.4392367128457488, 1.2068537943985003, 1.2068537943985003, 1.1769802675577083, 1.1769802675577083, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.5064526141464613, 0.4224794575588441, 0.4224794575588441, 0.4224794575588441, 0.4224794575588441, 0.4224794575588441, 0.4224794575588441, 0.4224794575588441, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753, 0.4058498765458753], [0.4265122218971614, 0.4265122218971614, 0.4265122218971614, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716, 0.4021707224897716], [1.1758156184698911, 1.1758156184698911, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 1.0967618606153287, 0.6449396689839192, 0.6449396689839192, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.4935577918349121, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316, 0.44906193342424316], [1.1279104459309421, 1.1279104459309421, 1.1279104459309421, 1.1279104459309421, 1.1279104459309421, 1.0875315191838308, 1.0875315191838308, 0.7347135241866921, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834, 0.39597815157574834], [1.2681644048875433, 0.987459082771941, 0.987459082771941, 0.987459082771941, 0.987459082771941, 0.987459082771941, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.6559536643809205, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044, 0.43301806574933044], [0.7930150042421884, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126, 0.41858518653001126]])
gA_random = torch.tensor([[1.89580954, 2.34302652, 1.90421585, 3.04651037, 2.43974943,
       2.87411693, 1.17758548, 1.80837219, 1.43768088, 1.76557936,
       1.46092081, 1.4837276 , 1.5887632 , 1.84640938, 1.67226971,
       3.34358453, 1.48327134, 1.35459335, 2.38632371, 1.67475688,
       2.77066638, 1.42638113, 1.22648259, 1.94779992, 1.71297984,
       1.29640213, 0.9818017 , 2.23493544, 1.95831733, 1.88084132,
       1.75799963, 2.63200947, 3.042008  , 2.34526796, 1.43598873,
       3.11661373, 1.13851534, 2.36408146, 1.28628408, 2.43378367], [1.32280491, 2.69304161, 1.82716117, 2.75571573, 2.49879424,
       2.01662325, 1.7766041 , 1.43042692, 1.71559215, 1.38426143,
       1.80994593, 2.08416923, 1.52443353, 1.55157071, 1.09209915,
       3.17652727, 2.08521023, 2.99334221, 2.10791883, 1.24566791,
       2.31492585, 3.32174476, 2.31198196, 1.42780417, 1.23886101,
       1.58599384, 1.49804475, 2.99879436, 2.18235149, 1.20335061,
       1.66499037, 2.5767663 , 2.69508537, 1.25879861, 1.84895409,
       2.02565269, 1.18044705, 1.70518491, 3.35066853, 2.62207795], [2.003175  , 2.5745471 , 2.5712551 , 1.52952183, 1.46935182,
       2.81855873, 1.07315663, 2.0387564 , 1.09365063, 2.29605189,
       1.62698633, 3.87815459, 1.4545904 , 1.39055406, 1.16470466,
       1.32817343, 1.72666559, 2.26056272, 1.77405255, 2.75798788,
       1.9340158 , 1.78982626, 3.36082812, 1.79998055, 1.78864443,
       2.25353943, 1.45438796, 2.58030372, 1.29883772, 1.3239715 ,
       1.3535533 , 2.92771   , 2.6821663 , 1.24934089, 1.37921876,
       1.64318454, 1.46833382, 1.9624486 , 2.67553857, 2.80693456], [1.70029331, 1.66878012, 2.50323454, 2.13987294, 2.18453425,
       1.2362411 , 2.81634668, 1.79561706, 4.02679243, 2.92734191,
       2.45930572, 2.46507852, 1.70012196, 1.92423407, 1.58630081,
       1.57687093, 1.34842633, 2.04390692, 1.54300477, 1.44650432,
       1.97052103, 3.13581756, 3.19017614, 1.13203095, 1.99100617,
       1.54205906, 2.27610684, 2.12747648, 2.576727  , 1.20555876,
       2.06023614, 2.47068126, 1.61955245, 1.73925075, 3.014354  ,
       1.19685434, 2.7594817 , 1.53142707, 2.82703339, 1.50820736], [3.01941425, 2.24118014, 2.21954788, 2.21013834, 2.94920398,
       1.26484461, 1.98667854, 1.97766595, 1.87887999, 2.44060499,
       1.26113232, 2.29487833, 1.51543468, 2.13275829, 1.37365057,
       1.45237088, 2.27101094, 2.60193624, 1.94558731, 2.01517859,
       1.49096084, 1.36234608, 2.99463856, 2.13022738, 3.15402441,
       2.06005749, 1.14791425, 2.29895236, 2.32049475, 1.36289675,
       1.79402816, 2.192518  , 3.44411482, 2.00707339, 2.8406201 ,
       1.73396719, 2.95197727, 2.72546775, 2.80871562, 1.98900303], [1.25613637, 1.92233924, 3.02680797, 1.05475743, 2.07574526,
       2.489433  , 1.63540409, 3.0140748 , 1.61541904, 3.37503658,
       2.24641056, 2.17409788, 1.5158473 , 1.84825534, 2.39881452,
       1.85374355, 2.24334216, 1.73234651, 1.32776028, 2.57199736,
       1.6597163 , 2.72197225, 2.20420059, 1.62727467, 2.10745538,
       1.79644989, 1.45355914, 1.6559017 , 2.58548074, 1.60486753,
       1.40805921, 3.36346745, 1.61905179, 1.6914237 , 1.71632055,
       2.85927872, 1.83276344, 1.04038624, 1.92943121, 2.41433603], [2.21291583, 2.34302712, 1.96911745, 2.96915837, 2.76889949,
       2.35765873, 2.07200723, 1.76620389, 3.25410889, 1.69881266,
       2.49822399, 2.08329815, 1.21482152, 2.03286958, 1.36120386,
       2.74582726, 1.63195983, 1.79330883, 1.92439825, 2.48101305,
       1.98443165, 1.48009859, 2.51870901, 1.38306099, 1.94669659,
       2.20104389, 1.35134397, 1.25820888, 2.2268117 , 3.69394779,
       1.7826328 , 1.9971185 , 2.0228855 , 1.21432522, 2.85634028,
       1.53924662, 1.63157584, 1.62710497, 1.33221308, 1.43466806], [2.15655387, 3.10267959, 2.49049322, 2.35124704, 2.47176218,
       2.10732968, 2.87714531, 1.36030535, 1.13891448, 3.0644881 ,
       1.38593559, 1.74531421, 1.63202603, 2.57242587, 2.64583708,
       1.82631116, 1.56408879, 2.36167085, 2.52245236, 2.44421675,
       1.46517866, 1.07224029, 1.94541389, 1.87280175, 1.06528973,
       1.32980472, 1.38660013, 2.97434123, 1.76601951, 2.48458079,
       1.24034491, 2.21109604, 1.71773444, 2.46837499, 1.41824524,
       2.02404163, 1.16134187, 3.49897518, 2.50077989, 2.29075061],[2.50315646, 1.65995225, 2.1453555 , 3.96718525, 2.31841281,
       2.85701977, 1.49219697, 3.36567514, 2.7273895 , 1.40981935,
       2.2874855 , 1.70579988, 2.22616594, 2.32163159, 2.74201392,
       1.86272784, 1.37815492, 2.36600217, 2.11294144, 2.75539236,
       1.08443497, 2.39675117, 1.85007414, 1.60574169, 1.67348458,
       1.463883  , 1.34783433, 2.40182059, 2.39553774, 2.41811705,
       2.68322274, 2.73769503, 1.71959369, 2.00084495, 1.61774536,
       1.61152711, 2.08534279, 2.45762082, 3.19870746, 2.46177704], [1.69663482, 1.23273835, 1.60145706, 2.88400724, 1.4570615 ,
       1.32642411, 1.23704856, 2.67840386, 1.20504349, 1.81271015,
       1.18494448, 1.44005526, 3.23966612, 3.66005451, 2.97187519,
       1.94014364, 1.62962105, 2.63842981, 2.64895353, 2.61924128,
       3.28488366, 2.55012404, 1.62194249, 3.03048009, 2.07102794,
       1.92723014, 1.09892522, 1.65732534, 1.71277217, 3.18447736,
       1.20872299, 2.17206854, 1.84982545, 1.645935  , 2.07582295,
       1.4384013 , 2.32535232, 1.60127886, 1.98512002, 1.66708444]])
cycL_bo = torch.arange(0, 830, 21)
best_bo = torch.tensor([[0.8880511909254067, 0.7603510528546753, 0.7603510528546753, 0.6445928909977148, 0.6445928909977148, 0.6445928909977148, 0.6445928909977148, 0.5297948533697334, 0.5297948533697334, 0.5297948533697334, 0.4100624043386106, 0.4100624043386106, 0.4100624043386106, 0.4100624043386106, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513, 0.38866856924446513], [0.7221741712041486, 0.7221741712041486, 0.7221741712041486, 0.6017710092824122, 0.6017710092824122, 0.6017710092824122, 0.6017710092824122, 0.4188652358563724, 0.4188652358563724, 0.4188652358563724, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4102183670523987, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294, 0.4013613371413294], [0.8497352337114688, 0.8217204022686435, 0.8217204022686435, 0.5199718072668861, 0.5199718072668861, 0.5199718072668861, 0.5199718072668861, 0.5199718072668861, 0.5199718072668861, 0.5199718072668861, 0.41237673927805435, 0.41237673927805435, 0.41237673927805435, 0.41237673927805435, 0.41237673927805435, 0.41237673927805435, 0.41237673927805435, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644, 0.3931315919764644], [0.8741011553118034, 0.7724954280982703, 0.7724954280982703, 0.5730667446321593, 0.5730667446321593, 0.5730667446321593, 0.5730667446321593, 0.45034969511732664, 0.45034969511732664, 0.45034969511732664, 0.45034969511732664, 0.45034969511732664, 0.44722656702593866, 0.44722656702593866, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627, 0.40151364852666627], [0.9301044326781653, 0.8297837511764771, 0.8297837511764771, 0.44371505542940615, 0.44371505542940615, 0.44371505542940615, 0.44371505542940615, 0.4164826625148886, 0.4164826625148886, 0.4164826625148886, 0.4151048805574145, 0.4151048805574145, 0.4151048805574145, 0.4151048805574145, 0.4151048805574145, 0.4151048805574145, 0.4151048805574145, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144, 0.39099936323078144], [0.801671775999754, 0.7588026756785822, 0.7588026756785822, 0.48159576532987586, 0.48159576532987586, 0.48159576532987586, 0.48159576532987586, 0.48159576532987586, 0.48159576532987586, 0.48159576532987586, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.41181965150548683, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133, 0.3938584332902133], [0.8350160694904816, 0.7558967367823972, 0.7558967367823972, 0.47499785148285545, 0.47499785148285545, 0.47499785148285545, 0.47499785148285545, 0.4197596339101285, 0.4197596339101285, 0.4197596339101285, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4122703899785328, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374, 0.4017056011500374], [0.8885358087294402, 0.7547188592304784, 0.7547188592304784, 0.6909766310693851, 0.6909766310693851, 0.6909766310693851, 0.6909766310693851, 0.48677785896953707, 0.48677785896953707, 0.48677785896953707, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.41116144765055374, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248, 0.4037280730498248], [0.918270279451012, 0.6961121901982426, 0.6961121901982426, 0.5611626977799653, 0.5611626977799653, 0.5611626977799653, 0.5611626977799653, 0.41877388880575395, 0.41877388880575395, 0.41877388880575395, 0.4103551719677614, 0.4103551719677614, 0.4103551719677614, 0.4103551719677614, 0.4103551719677614, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426, 0.38943435344050426], [0.8454544864411714, 0.8429534613357984, 0.8429534613357984, 0.5032866168373051, 0.5032866168373051, 0.5032866168373051, 0.5032866168373051, 0.5032866168373051, 0.5032866168373051, 0.5032866168373051, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.4131288214493376, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317, 0.3926542642569317]])
gA_bo = torch.tensor([[1.61899002, 1.54174548, 3.00172245, 1.51058818, 2.09603771,
       1.86036056, 2.88522446, 1.14106774, 2.72267296, 2.16844966,
       1.14091684, 0.91406922, 1.03731131, 1.31204966, 0.97267764,
       0.99719382, 1.80823684, 1.6894563 , 2.76645246, 2.08143818,
       1.11757169, 1.69721442, 4.3748272 , 1.09493644, 1.40036916,
       1.98810134, 1.20260729, 1.20453864, 1.28608834, 1.83307175,
       1.49316802, 2.40079564, 1.55063943, 1.70521182, 1.36562029,
       1.7979422 , 1.20676708, 1.46832048, 1.82101671, 1.81424213], [1.4959782 , 1.61241574, 2.74883359, 1.22522212, 2.02591546,
       1.85999147, 2.98650396, 1.18688127, 2.41673899, 1.96172378,
       1.16682819, 1.05004338, 0.97836057, 1.44461353, 1.14167654,
       1.47872406, 1.51043007, 1.07195701, 1.76945292, 2.1509394 ,
       1.02428432, 1.6920936 , 1.5079545 , 1.27735882, 4.43286842,
       1.20819153, 1.3801489 , 1.89608857, 2.80439651, 1.40634481,
       1.76185552, 2.4139924 , 1.69739376, 1.62137444, 1.497939  ,
       3.63609951, 2.27167636, 2.57549775, 1.56826057, 1.25386149], [1.5796061 , 1.44348814, 2.8582149 , 1.22356553, 2.17442096,
       1.79016905, 2.97335863, 1.22453649, 2.51877946, 2.27768625,
       1.1907907 , 1.22155365, 1.3006602 , 0.97810879, 1.3505593 ,
       1.69000776, 2.02855461, 1.11396337, 2.69584976, 1.98558228,
       0.94978557, 1.16728755, 4.38398803, 1.32774586, 1.78937188,
       1.48063639, 1.21179441, 1.51982703, 1.29976424, 1.09515159,
       2.62985795, 1.57898218, 2.01484332, 1.59775943, 1.31395174,
       1.90990015, 3.34731361, 2.21898435, 2.09446766, 1.40615313],[1.52709432, 1.48848544, 3.03048022, 1.41539917, 1.99263802,
       1.89340915, 3.19358266, 1.29365892, 2.76125392, 2.13030633,
       1.1882145 , 1.13613804, 1.19987549, 1.18289572, 1.03027978,
       1.06111705, 1.97585895, 1.45965785, 1.98434201, 1.36046074,
       1.61187154, 4.37959659, 1.03112192, 1.46171196, 2.66671722,
       1.29141459, 1.92853838, 1.20400022, 1.69805056, 1.71406122,
       1.21467061, 1.586235  , 2.58172973, 1.41804139, 1.26070922,
       2.02274704, 2.50889889, 2.05124026, 3.59431622, 2.414805  ], [1.74371538, 1.53315898, 2.82122739, 1.25747822, 2.06116852,
       2.0345162 , 3.05180546, 1.14021977, 2.67632928, 2.24552752,
       1.13182333, 0.93551944, 1.47075106, 1.06748826, 1.32993459,
       1.74756943, 1.24303938, 1.03877044, 1.84664466, 2.09159426,
       1.42244346, 1.21574044, 1.34988847, 2.73800036, 4.5081806 ,
       1.82276974, 1.30225733, 1.50737915, 1.29711695, 1.2561038 ,
       1.12675651, 1.50822114, 2.4417978 , 1.77222109, 1.44034188,
       1.29458944, 2.06194412, 1.82994873, 1.30809153, 1.12010203], [1.55002847, 1.34075833, 2.71194933, 1.2469046 , 1.77313757,
       1.94288123, 2.84053324, 1.41641438, 2.62048664, 2.06631031,
       1.13624399, 1.2432773 , 0.9599985 , 1.07367659, 1.7690382 ,
       1.22872339, 1.76808215, 1.58543665, 1.10970502, 1.05056602,
       2.03160432, 2.7207268 , 1.10559259, 1.23011827, 4.32243023,
       1.34887979, 1.68045378, 1.58793856, 1.13568015, 1.18226932,
       2.31785418, 1.7034159 , 2.45196896, 1.6955261 , 1.3652913 ,
       2.04432363, 2.31449645, 2.53985229, 2.03931242, 1.07256592], [1.51512426, 1.54821156, 2.94365578, 1.4201919 , 2.11306186,
       1.67873096, 2.79115151, 1.08432667, 2.47614748, 1.99225132,
       1.01488727, 0.98619131, 1.44986839, 0.99820164, 1.13799193,
       1.42518937, 2.004143  , 1.56432615, 0.98705608, 1.97726522,
       1.30015525, 1.06693483, 1.09965227, 1.40456977, 2.82410251,
       1.36443656, 1.67115089, 4.52451552, 1.84864629, 1.3650215 ,
       2.35322014, 1.24412879, 1.69338159, 2.18999362, 1.48389949,
       1.32204506, 2.39034845, 1.75123416, 3.50404064, 1.26573419], [1.56102282, 1.52216023, 2.86710945, 1.45607552, 2.02056198,
       1.97022096, 2.84917214, 1.1329608 , 2.41741538, 2.01714374,
       1.12298559, 1.16612621, 1.1078336 , 1.37472475, 1.56380177,
       1.29773753, 1.97651189, 1.5177599 , 1.17482635, 4.45628688,
       2.87736567, 1.42176343, 1.01561181, 2.05420488, 1.68112353,
       2.0841822 , 1.26425039, 1.21363921, 1.22416317, 1.40847436,
       2.45428183, 1.687369  , 1.73798456, 1.70007417, 1.58657807,
       1.097536  , 1.98797993, 2.13372466, 1.51865556, 2.48391529], [1.69489931, 1.42149287, 3.10052719, 1.23712886, 2.09115468,
       1.8662683 , 2.89558799, 1.08514459, 2.4809488 , 2.00192429,
       1.03982288, 1.04808241, 1.04748024, 1.38570439, 1.22201551,
       0.90721965, 1.13645536, 2.13895034, 1.16473332, 1.57763817,
       2.63322714, 1.11856027, 2.06500701, 1.06487883, 1.81111131,
       1.41571393, 4.39885738, 1.30820178, 1.32303884, 1.37626244,
       2.3828678 , 1.5274186 , 1.44425908, 1.55685763, 1.6394571 ,
       1.82600063, 2.41859294, 2.42188869, 1.07401029, 1.9968895 ], [1.6262323 , 1.43316263, 3.05830631, 1.2038714 , 2.21009521,
       1.94943149, 2.90829356, 1.31615062, 2.43164284, 2.20688116,
       1.17126433, 1.25269479, 1.03826128, 1.34377212, 1.11131174,
       1.76237652, 1.5960128 , 1.55160184, 1.00829658, 2.20377204,
       2.75122344, 0.99948116, 1.49106206, 4.49248419, 1.22943479,
       1.15397664, 1.11961623, 1.3261558 , 1.64815973, 0.82477874,
       1.68923451, 1.89838841, 1.37128336, 1.60947056, 2.46627317,
       1.66585387, 2.14229648, 3.52876236, 2.51045229, 1.61953878]])
cycL_es = torch.arange(0, 840, 10)
best_es = torch.tensor([[0.4777185015558228, 0.4777185015558228, 0.4777185015558228, 0.4777185015558228, 0.4777185015558228, 0.4777185015558228, 0.4777185015558228, 0.4768958477559424, 0.47368308531390735, 0.4708461868352877, 0.4679066305603926, 0.46544480272216143, 0.4630035417626578, 0.4609976631280813, 0.4589785221840025, 0.4572904530797276, 0.4572904530797276, 0.4572904530797276, 0.45707738155510325, 0.4535057735663986, 0.4503696611106364, 0.44754251059715666, 0.44484133361116707, 0.4427107224491218, 0.4404527291241048, 0.4384129241019278, 0.43643430827253593, 0.4329657948129245, 0.42948801825018157, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405, 0.42789729157008405], [0.47712672975628234, 0.47712672975628234, 0.47712672975628234, 0.47712672975628234, 0.47712672975628234, 0.47712672975628234, 0.47712672975628234, 0.4769258954628558, 0.47358342823002786, 0.47065930346748225, 0.46780929009265154, 0.46535845851407, 0.4636246550752088, 0.46089687907009425, 0.4590664669319593, 0.4573376813115106, 0.4573376813115106, 0.4573376813115106, 0.4569709918340889, 0.4536042097985624, 0.451301117716412, 0.44751895439801653, 0.44482723319175155, 0.4426130554897181, 0.4404322272029036, 0.4387503363768284, 0.43680583536253137, 0.43275816651890453, 0.43128042099805003, 0.4287453560266993, 0.4287453560266993, 0.4287453560266993, 0.4287453560266993, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713, 0.4179253454357713], [0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.47721534236572505, 0.4768624865048844, 0.4735583484470629, 0.47060414539330014, 0.46784479170952187, 0.46529089409590707, 0.46299366770058514, 0.4608963921942967, 0.4591494922085614, 0.45722631933842284, 0.45722631933842284, 0.45722631933842284, 0.45705311174687946, 0.45352593440753763, 0.45045331458326104, 0.44749803479576716, 0.4451248498912854, 0.4424206937131251, 0.4404200182943627, 0.43841266231310744, 0.4363789577356977, 0.43276356561164875, 0.4297219698218451, 0.4290387901801694, 0.4290387901801694, 0.4290387901801694, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383, 0.42087481103342383], [0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846, 0.47744677857672846], [0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335, 0.4772293642237335], [0.4776545996119582, 0.4776545996119582, 0.4776545996119582, 0.4776545996119582, 0.4776545996119582, 0.4776545996119582, 0.4776545996119582, 0.47683298256742546, 0.47354869934546184, 0.47078591741004194, 0.4678894802238194, 0.46544646201906337, 0.4631888144970223, 0.461201997830087, 0.45950129501819637, 0.45723243464263424, 0.45723243464263424, 0.45723243464263424, 0.4569560389414695, 0.453552913073092, 0.4503376195364401, 0.4479049342425071, 0.4449188755896477, 0.4424512557440894, 0.4402554118220658, 0.4382277606235023, 0.43662266831244767, 0.4328252074274333, 0.4302426588984616, 0.4274751092191911, 0.4274751092191911, 0.4274751092191911, 0.4274751092191911, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294, 0.41810257570539294], [0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418, 0.4782402706243418], [0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4771357538147768, 0.4737490304132005, 0.4705746804819669, 0.46780011082565504, 0.46529322063760414, 0.46305773392976646, 0.4608958332434202, 0.45932646857065207, 0.457225479148215, 0.457225479148215, 0.457225479148215, 0.45698908357308027, 0.4538722328785843, 0.4503765663232385, 0.4474590528516935, 0.4448535176979864, 0.442529296190595, 0.44071022010708544, 0.43864054882898706, 0.4364055466249085, 0.4327925629187835, 0.42959353326664296, 0.4279875277102869, 0.4279875277102869, 0.4252089434154309, 0.4252089434154309, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038, 0.4181667265211038], [0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47712240039759524, 0.47354471801699033, 0.4705436221408915, 0.46810351392565586, 0.4654220457447221, 0.4629944965652159, 0.4610187338702623, 0.45899789573735184, 0.4572985348858314, 0.4572985348858314, 0.4572985348858314, 0.45698066030612267, 0.45351670111976705, 0.45042235864278324, 0.4477711247200154, 0.444828721793936, 0.4425243125495672, 0.44023615437956815, 0.43823784059816195, 0.43668299591715387, 0.43280785299404945, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324, 0.4294199295773324], [0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846, 0.47767741443220846]])
gA_es = torch.tensor([[1.11545598, 0.99721754, 1.0146663 , 1.01285852, 1.04489007,
       1.03415768, 1.0343801 , 1.11935547, 0.99261529, 0.95512298,
       0.98092674, 1.25748436, 1.03085985, 1.23958763, 1.27112849,
       0.98687857, 0.83344412, 1.08803248, 0.98780767, 0.99683177,
       1.06907508, 1.09477137, 1.09708631, 1.03953038, 1.05375127,
       0.98701847, 1.15322704, 1.11151427, 1.12328362, 1.11302538,
       1.28717536, 1.20001563, 1.1842371 , 1.31125652, 1.34193056,
       1.08640211, 1.33754013, 1.23794603, 1.47041596, 1.38317202,
       1.47964017, 1.31301198, 1.53486283, 1.13479924, 1.53461216,
       1.51014495, 1.20033365, 1.41963293, 1.44831923, 1.4368941 ,
       1.32663413, 1.33482397, 1.46979686, 1.32743993, 1.3788745 ,
       1.3553844 , 1.50990633, 1.38107962, 1.53118816, 1.29641883,
       1.47375712, 1.56402697, 1.20594504, 1.3437211 , 1.50084212,
       1.36379087, 1.53538465, 1.45836045, 1.37442968, 1.2887663 ,
       1.36976797, 1.24483629, 1.39700886, 1.46464047, 1.67538506,
       1.47139557, 1.58913894, 1.67064702, 1.35741722, 1.54726884,
       1.4229677 , 1.51538449, 1.47172312, 1.29394489], [0.99640179, 0.97897012, 1.09747739, 1.14882152, 1.06300317,
       1.21129941, 0.89154638, 0.98088666, 1.25285861, 1.00136552,
       1.05982165, 1.19343079, 0.98150119, 1.08296561, 1.10914572,
       1.01072974, 1.12599795, 1.04770652, 1.01922364, 1.11129174,
       1.15896361, 0.87387223, 1.10082221, 1.00521232, 1.00335657,
       1.13818868, 0.88553533, 1.1104011 , 1.00585725, 1.24905868,
       1.23504611, 1.08339518, 1.25549025, 1.14212657, 1.11448692,
       1.33950677, 1.41330865, 1.2601661 , 1.42308704, 1.56101215,
       1.36354124, 1.60462265, 1.43421027, 1.41410139, 1.4214127 ,
       1.40653053, 1.37675133, 1.48508647, 1.48699248, 1.45682102,
       1.51684436, 1.51533664, 1.54413448, 1.27686435, 1.46192304,
       1.54118913, 1.53406468, 1.47370344, 1.44487293, 1.4598435 ,
       1.47462388, 1.40967014, 1.46796452, 1.31126421, 1.27037766,
       1.43382863, 1.50315217, 1.42367477, 1.32702311, 1.40966552,
       1.45799679, 1.2264631 , 1.4451573 , 1.36461734, 1.4111919 ,
       1.3192135 , 1.52319819, 1.36862884, 1.40843563, 1.47223202,
       1.60213325, 1.38289563, 1.49825317, 1.37918572], [1.13264286, 0.84083948, 1.13470528, 1.01945579, 0.96077013,
       1.08675499, 1.05023961, 1.11562021, 0.99334163, 0.84314699,
       1.24802824, 1.21432306, 1.07517112, 1.05143559, 1.12500622,
       1.15110303, 1.0889439 , 0.96847518, 0.81598651, 0.95983793,
       1.06900001, 1.10526944, 1.22336312, 0.98704471, 1.12482672,
       0.96601278, 1.14249939, 1.09830215, 1.29176236, 1.14081321,
       1.013334  , 1.17220172, 1.04064162, 1.04575712, 1.23247747,
       1.22839652, 0.91937572, 1.3073588 , 1.15557966, 1.08622392,
       1.08697895, 1.10762632, 1.16063457, 1.02069411, 1.16345042,
       1.33837318, 1.25506118, 1.14958592, 1.08106513, 1.08887803,
       1.20022713, 0.87225718, 1.21971731, 1.2524343 , 1.34218555,
       1.19059545, 1.29087333, 1.26535402, 1.43591674, 1.20235686,
       1.13564024, 1.12388062, 1.4092743 , 1.0654406 , 1.26546784,
       1.30683347, 1.3980816 , 1.32976505, 1.25301474, 1.38956072,
       1.59699587, 1.37127773, 1.53767503, 1.49044346, 1.37328589,
       1.49867207, 1.47517199, 1.43641212, 1.49270299, 1.46661861,
       1.51950968, 1.5418538 , 1.58764729, 1.38896101], [1.23934605, 1.12626963, 0.91280052, 1.21519274, 1.18423966,
       0.90906596, 1.2699194 , 1.19099419, 1.06407086, 0.98566185,
       1.20490175, 1.19372385, 0.91509611, 1.04228648, 1.06052304,
       1.14935627, 0.87662898, 1.1124224 , 1.24634193, 1.00439099,
       1.01373664, 1.20741556, 1.17324331, 1.13588868, 0.9650963 ,
       1.16716985, 1.2027639 , 0.94590036, 1.08085822, 0.99603001,
       1.04929521, 0.92940062, 1.08594572, 1.20959705, 1.20518265,
       1.04699994, 0.96167237, 1.17713547, 1.15579553, 1.36984887,
       1.17160436, 1.1406714 , 1.02540694, 0.99969715, 1.16664232,
       1.14470994, 1.08103949, 1.10510327, 1.05985408, 0.98920332,
       1.12102917, 0.94519256, 1.11141771, 1.05883585, 0.87586821,
       1.18785256, 0.92359525, 1.15730903, 1.06330931, 1.0258775 ,
       1.0266624 , 1.14320909, 1.13580972, 0.82445986, 1.17836792,
       1.17152826, 1.05851805, 1.08741419, 1.15444609, 1.13722585,
       1.21380756, 1.20716914, 0.990866  , 1.07279065, 0.9047286 ,
       1.14401917, 1.2144625 , 1.00278908, 1.19637735, 1.08513982,
       1.04070907, 1.13559784, 1.07675533, 0.88581339], [1.22018766, 0.96028505, 1.10693337, 1.10058937, 1.19891652,
       1.14746436, 1.09892709, 1.00446343, 1.28042627, 1.02961102,
       1.18814263, 0.88352737, 1.05567027, 1.10577097, 1.07898009,
       1.05217694, 1.05126473, 1.1106257 , 1.19466579, 1.12161269,
       1.2189061 , 1.14689788, 1.24051514, 0.88974264, 1.07333592,
       1.00071946, 1.21548657, 1.17252444, 1.14940543, 1.00023361,
       1.00872419, 0.99789734, 1.04352461, 0.88655499, 1.14569895,
       0.91083927, 1.15886855, 1.03393434, 1.13033077, 1.00867733,
       1.01672373, 1.04459221, 1.02298599, 1.29480097, 1.09161626,
       1.02736789, 1.07650287, 0.99534497, 1.14820033, 0.94745448,
       1.16699646, 0.98605474, 0.88892443, 1.02289263, 1.0441132 ,
       1.16763165, 1.23629127, 1.15399581, 1.14603959, 1.10449102,
       0.89746504, 1.05521488, 0.98836889, 1.15483699, 0.97780311,
       1.06730522, 1.04261508, 1.11862284, 1.1248687 , 0.97828557,
       0.97581856, 1.16239534, 1.21840376, 0.99821479, 0.9760503 ,
       0.91508119, 1.28423939, 1.10625998, 0.86720289, 1.06801585,
       0.96700896, 1.00443315, 1.0409278 , 1.3130805 ], [1.12011663, 0.93508825, 0.90455798, 1.0175817 , 1.05815952,
       1.05694017, 1.24707145, 1.03822946, 1.17336538, 0.98179205,
       1.00257176, 1.14163187, 1.04588499, 1.01862832, 1.1208366 ,
       0.86419453, 1.01656889, 1.06133634, 1.12549549, 1.13365859,
       1.11441674, 0.98271977, 1.11881578, 1.06723584, 1.28552717,
       1.10437674, 1.11126889, 1.07191139, 1.16904131, 1.02137035,
       1.30294392, 1.32200414, 1.28600046, 1.0945888 , 1.36761227,
       1.4001053 , 1.42444287, 1.40524882, 1.34856556, 1.58693602,
       1.36168465, 1.17020386, 1.39993963, 1.40504228, 1.38166907,
       1.26188113, 1.32273984, 1.43296663, 1.36250598, 1.32933387,
       1.48517574, 1.35925579, 1.19986044, 1.4207051 , 1.34261891,
       1.33676667, 1.27403107, 1.55230136, 1.40013289, 1.45770661,
       1.52077783, 1.24182474, 1.43845971, 1.35162818, 1.4766756 ,
       1.41630289, 1.37050623, 1.28512813, 1.27466559, 1.27053394,
       1.29940889, 1.20234048, 1.31405017, 1.39323221, 1.38060783,
       1.4438246 , 1.39407184, 1.29993243, 1.40790524, 1.29815385,
       1.58221864, 1.34306746, 1.5593821 , 1.41772066], [1.20862278, 0.92772693, 0.89838775, 1.19302791, 1.3809424 ,
       1.00570791, 1.23363389, 1.22060271, 1.07619816, 0.88905646,
       1.18352981, 1.02498788, 1.03394447, 1.02784228, 1.19188594,
       1.05175373, 1.06796371, 1.0303991 , 1.08938275, 1.18243781,
       1.15652365, 0.95867076, 1.16054535, 1.06622211, 1.36132867,
       1.05044587, 1.05419577, 1.17448512, 1.06073376, 0.98251092,
       1.08876808, 1.25242971, 1.2220502 , 1.07876852, 0.9164988 ,
       1.18297202, 1.02084723, 0.9001619 , 1.1464857 , 1.06208348,
       1.02713428, 1.0791836 , 1.12291639, 1.06965579, 1.18803678,
       1.18300532, 0.99215157, 1.12721984, 1.01515031, 0.96022469,
       1.07023583, 0.93245254, 0.95060986, 1.09472608, 1.11942839,
       1.01013318, 0.94199711, 1.14431989, 0.95244355, 1.23791179,
       1.13307375, 1.23938348, 1.24605256, 1.28598373, 1.12643018,
       0.96722216, 1.1328057 , 1.1433802 , 1.07087004, 1.05668503,
       1.14067693, 1.1115068 , 1.1108817 , 1.05672239, 1.17766533,
       1.10776949, 0.86476269, 1.20944195, 1.20592438, 0.85071265,
       1.1779546 , 1.22935628, 0.95892925, 0.95020761], [1.02833391, 1.04336712, 1.22282897, 1.07802721, 1.18758948,
       0.89591648, 0.99846677, 1.22533085, 1.13011524, 1.12332196,
       1.09954026, 1.12999445, 1.11323191, 1.16660014, 0.83741244,
       1.18867262, 1.15215672, 0.96127951, 1.06791766, 1.22406247,
       0.99580254, 1.0261544 , 1.09794828, 1.17250485, 1.16684784,
       0.9072571 , 1.11555188, 1.03787793, 0.93735547, 0.99358234,
       1.03042924, 1.04381551, 1.20411891, 0.83221621, 1.05498697,
       0.84573755, 1.00397955, 0.94018434, 1.10847847, 1.22384845,
       1.1558374 , 1.07006664, 1.35178208, 1.23947724, 1.26808227,
       1.11581705, 1.36852908, 1.35436423, 1.3091422 , 1.37378652,
       1.28558866, 1.25948186, 1.25081294, 1.30712988, 1.12498259,
       1.39507036, 1.35147619, 1.2944665 , 1.22356837, 1.11348454,
       1.29252896, 1.33504673, 1.30105482, 1.42079856, 1.30792199,
       1.29556467, 1.50728829, 1.06249063, 1.40323418, 1.40906736,
       1.3015508 , 1.37949776, 1.25620355, 1.12884395, 1.12950714,
       1.34297765, 1.37690687, 1.35672962, 1.18143479, 1.32972658,
       1.32321547, 1.15860452, 1.12740998, 1.18040904],[1.03417489, 1.17859695, 1.14445432, 1.05637545, 0.8963214 ,
       0.96198657, 0.99445903, 1.17918689, 1.08473877, 1.03113011,
       1.0838268 , 0.97981698, 1.17914199, 1.02534147, 0.91112563,
       1.08561049, 1.07998016, 0.98666053, 1.06239223, 1.35732827,
       1.1362622 , 1.02164503, 1.27235014, 1.14889082, 1.10625604,
       1.14289596, 1.01207084, 1.0321231 , 1.07969428, 1.10831631,
       0.91323783, 1.10230415, 1.01374839, 1.0684032 , 1.01640711,
       0.91491437, 1.04856671, 0.96216804, 1.07749206, 0.94782857,
       1.17343395, 1.1353747 , 1.033554  , 1.07128514, 1.14101832,
       1.17273863, 1.0015838 , 0.95209423, 1.04675307, 1.16578271,
       1.24967012, 1.41914657, 1.58496222, 1.28778883, 1.2507759 ,
       1.44233912, 1.35162423, 1.43948034, 1.39493753, 1.69057427,
       1.59690447, 1.45145353, 1.52312499, 1.57312684, 1.35815902,
       1.49190301, 1.66792634, 1.47278195, 1.50876904, 1.51957446,
       1.60605843, 1.53350511, 1.5754212 , 1.40168928, 1.55424337,
       1.53082269, 1.46642362, 1.51691392, 1.52274447, 1.46741112,
       1.49127204, 1.6254634 , 1.53121069, 1.45867323],[0.9545161 , 1.01319875, 1.00602236, 1.06144506, 1.12436997,
       1.14351007, 1.03736245, 1.02045099, 1.09445678, 1.03249266,
       1.06666771, 1.0777806 , 0.92658764, 1.0458149 , 1.00840184,
       1.19622114, 1.25275138, 1.1210322 , 1.08370814, 1.19444453,
       1.18102569, 1.01700834, 1.21116476, 0.98270351, 1.01283889,
       1.10230516, 0.91220392, 1.15800486, 1.08938587, 0.97894576,
       1.05550014, 1.00343822, 1.24225155, 1.15269133, 1.20413901,
       1.06040098, 1.12525564, 1.20454984, 1.17281675, 1.07168006,
       1.15592902, 1.07858924, 0.84573707, 1.00978412, 1.16129213,
       1.07487071, 1.00961104, 1.09193571, 1.19456062, 1.03543255,
       1.06136955, 1.0918307 , 1.02256792, 1.0625678 , 1.02258935,
       1.10109933, 1.02171569, 1.06588812, 1.14822878, 1.19311003,
       1.22767576, 0.96201083, 1.07144331, 1.20907225, 0.9167684 ,
       1.11267403, 1.1233756 , 1.04933797, 1.20672463, 0.90102767,
       1.03485141, 1.14633157, 1.00146008, 1.0607013 , 1.02533206,
       1.05981008, 1.0671676 , 1.12588003, 1.1064727 , 0.96072983,
       1.1408974 , 1.11863839, 1.12289624, 1.15282048]])


In [ ]:
figbv = 'best value'
c = (0.6328422475018423, 0.4747981096220677, 0.29070209208025455)
plt.figure(num = figbv, figsize=(18, 14))
plot_uq_(cycL, best_A_tot.numpy(), cm[0], r'Our method ' , optval=None, optgap=None,
                fignum=figbv)
plot_uq_(cycL_bo, best_bo, cm[2], r'BO/EI' , optval=None, optgap=None,
                fignum=figbv)
plot_uq_(cycL_random, best_random, cm[4], r'Random' , optval=None, optgap=None,
                fignum=figbv)
plot_uq_(cycL_es, best_es, cm[6], r'Expert System' , optval=None, optgap=None,
                fignum=figbv)
plt.ylim(0, 2.)
plt.ylabel(r'$C_{total}$ best value')
#plt.savefig('figures/cost_vscycles_paper.pdf')

# plt.tight_layout()

In [ ]:
c = (0.6328422475018423, 0.4747981096220677, 0.29070209208025455)
figbb = 'Growth'
plt.figure(num = figbb, figsize=(18, 14))
plot_uq_(cycL, gA_tot.numpy(), cm[0], r'Our method ' , optval=None, optgap=None,
                fignum=figbb)
plot_uq_(cycL_bo, gA_bo, cm[2], r'BO/EI' , optval=None, optgap=None,
                fignum=figbb)
plot_uq_(cycL_random, gA_random, cm[4], r'Random' , optval=None, optgap=None,
                fignum=figbb)
plot_uq_(cycL_es, gA_es, cm[6], r'Expert System' , optval=None, optgap=None,
                fignum=figbb)
#plt.savefig('figures/cost_vscycles_paper.pdf')
plt.ylabel(r'Normalized growth per cycle')
#plt.savefig('normalized_paper.pdf')

In [ ]:
cy = [[0.7253293688029772, 0.7253293688029772, 0.510889573922775, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.38116188078119384, 0.3519840732758098, 0.3519840732758098, 0.3519840732758098, 0.3519840732758098, 0.3519840732758098, 0.3519840732758098], [0.8998721689784005, 0.8998721689784005, 0.610947259853632, 0.42922400428103513, 0.42922400428103513, 0.42922400428103513, 0.42922400428103513, 0.42922400428103513, 0.41409575098970175, 0.41409575098970175, 0.41409575098970175, 0.41409575098970175, 0.41409575098970175, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.40877363411942536, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.4066633456359083, 0.40369749377698244, 0.3499836491470547, 0.3499836491470547], [0.8248961154621439, 0.8248961154621439, 0.527436499347619, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.38126957406289974, 0.35155765492763236, 0.35155765492763236, 0.35155765492763236, 0.35155765492763236, 0.35155765492763236, 0.35155765492763236], [0.9733673560822763, 0.9733673560822763, 0.6095174567105678, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811269076983761, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542, 0.3811005613768542], [0.9398089310533897, 0.9398089310533897, 0.601989958271294, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3814820435510393, 0.3800457255001526, 0.3800457255001526, 0.3800457255001526, 0.3800457255001526, 0.3800457255001526, 0.3800457255001526, 0.3524129317110535, 0.3524129317110535, 0.3524129317110535, 0.3524129317110535], [0.8073860231342601, 0.8073860231342601, 0.5996038006860784, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.3817075321360797, 0.37365790643830193, 0.37365790643830193], [0.7778965246895422, 0.7778965246895422, 0.5873321158380933, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.38120239376731446, 0.3740435063132481, 0.3740435063132481, 0.3518289377330816, 0.3518289377330816, 0.3518289377330816, 0.3518289377330816, 0.3518289377330816, 0.3518289377330816, 0.3518289377330816], [0.906236457075333, 0.906236457075333, 0.6367062349694327, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353, 0.38118650442574353], [0.8186753100472058, 0.8186753100472058, 0.6222094481179519, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3815364363695526, 0.3804955082462121, 0.3804955082462121, 0.3804955082462121, 0.3804955082462121, 0.3804955082462121, 0.3517372555554162, 0.3517372555554162, 0.3517372555554162, 0.3517372555554162, 0.3517372555554162, 0.3517372555554162, 0.3517372555554162], [0.8841105108469192, 0.8841105108469192, 0.5952330418034597, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551, 0.3812779056756551], [0.9633013659478895, 0.9633013659478895, 0.6150086250250291, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.38211132711516166, 0.3724145838776522, 0.3724145838776522, 0.3724145838776522, 0.350478081504771, 0.350478081504771], [0.950898723595052, 0.950898723595052, 0.6617764025649768, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904, 0.38171652595570904], [0.9490621457950233, 0.9490621457950233, 0.682304856281477, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.38145126414714303, 0.3805214000734153, 0.3805214000734153, 0.3805214000734153, 0.3805214000734153, 0.3805214000734153, 0.3805214000734153, 0.3534554254520933, 0.3534554254520933], [0.921757622377061, 0.921757622377061, 0.6291059952369339, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.38163912367398706, 0.3532418183608435, 0.3532418183608435, 0.3532418183608435, 0.3532418183608435, 0.3532418183608435], [0.8963918224609411, 0.8963918224609411, 0.6249069226907498, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.38157083093222066, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615, 0.35139091440415615], [0.916312756335758, 0.916312756335758, 0.6498465753239444, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3819514223794839, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.3780849780443606, 0.35247480217360505], [0.8409024020137692, 0.8409024020137692, 0.6085729749917473, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.38128675494665987, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676, 0.3777098886696676], [0.7847934352206954, 0.7847934352206954, 0.5921943555547697, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3813488117901136, 0.3531483219337551, 0.3531483219337551, 0.3531483219337551, 0.3531483219337551, 0.3531483219337551, 0.3531483219337551, 0.3531483219337551], [0.8980568202659658, 0.8980568202659658, 0.5575620287924214, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.38123123807431986, 0.3805187569528672, 0.3805187569528672, 0.3805187569528672, 0.3805187569528672, 0.3805187569528672, 0.3805187569528672, 0.3805187569528672, 0.3524690578825328, 0.3524690578825328, 0.3524690578825328, 0.3524690578825328, 0.3524690578825328, 0.3524690578825328], [0.8150844234326134, 0.8150844234326134, 0.47220612817035457, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.38198005803281593, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37814796482233937, 0.37626880727113954, 0.35194617539023954, 0.35194617539023954, 0.35194617539023954, 0.35194617539023954, 0.35194617539023954, 0.35194617539023954], [0.8999149193272894, 0.8999149193272894, 0.5815374682672246, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3813135099970551, 0.3520103986348362], [0.8996076367686295, 0.8996076367686295, 0.654162705601698, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.4064357703712633, 0.3819921486477602, 0.3819921486477602, 0.3819921486477602, 0.3819921486477602, 0.3819921486477602, 0.3819921486477602, 0.3819921486477602, 0.3511460609293562, 0.3511460609293562, 0.3511460609293562, 0.3511460609293562], [0.8886684147955333, 0.8886684147955333, 0.6133670412365789, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434, 0.38113927357852434], [0.9046469801113496, 0.9046469801113496, 0.5896196672219871, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.38133741344701705, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203, 0.37842698772058203], [0.7981998814486458, 0.7981998814486458, 0.5830434205047395, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.3811382499543907, 0.35159758873768165, 0.35159758873768165, 0.35159758873768165], [0.7765335644412522, 0.7765335644412522, 0.6123578422779503, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.3811379655097169, 0.35134336588769516, 0.35134336588769516, 0.35134336588769516], [0.877478562755378, 0.877478562755378, 0.46839618685729373, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452, 0.3811473301222452], [0.9655116314662449, 0.9655116314662449, 0.5305459843591976, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3814001661172336, 0.3793878978708112, 0.3793878978708112, 0.3793878978708112, 0.3793878978708112, 0.3793878978708112, 0.3524441012516575, 0.3524441012516575, 0.3524441012516575, 0.3524441012516575, 0.3524441012516575, 0.3524441012516575], [0.8106128595300013, 0.8106128595300013, 0.6461125792198696, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.3812079049446362, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445, 0.38083123357754445], [0.7645079091093264, 0.7645079091093264, 0.5829029151293299, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.3816692402475799, 0.36084716474863204, 0.36084716474863204, 0.36084716474863204, 0.36084716474863204, 0.36084716474863204, 0.36084716474863204], [0.8339576300790512, 0.8339576300790512, 0.5090045325814085, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3811971503085858, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788, 0.3808607224404788], [0.8798283913369824, 0.8798283913369824, 0.6469062503385475, 0.4232124242624802, 0.4232124242624802, 0.4232124242624802, 0.4232124242624802, 0.4232124242624802, 0.41418638720786105, 0.41418638720786105, 0.41418638720786105, 0.41418638720786105, 0.41418638720786105, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40882436222615637, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40688425125285227, 0.40028297251535644], [0.8482306878478555, 0.8482306878478555, 0.6340212631937847, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.3811258294585052, 0.37440135674511277, 0.37440135674511277, 0.3516363714404932], [0.8138592383012018, 0.8138592383012018, 0.5441857201381419, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3817647722245467, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.3784001180631325, 0.35335850477303016, 0.35335850477303016], [0.9455642050115706, 0.9455642050115706, 0.5897946946415662, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.38117955633145717, 0.3514740758654009], [0.9415498393249924, 0.9415498393249924, 0.6330868769332261, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.38127369282958934, 0.37931558858514236, 0.37931558858514236, 0.37931558858514236, 0.37931558858514236, 0.37931558858514236, 0.37931558858514236, 0.37931558858514236, 0.3521283632610059, 0.3521283632610059, 0.3521283632610059, 0.3521283632610059, 0.3521283632610059], [0.902675530115166, 0.902675530115166, 0.5766840458419602, 0.42426612642139433, 0.42426612642139433, 0.42426612642139433, 0.42426612642139433, 0.42426612642139433, 0.4149055711669669, 0.4149055711669669, 0.4149055711669669, 0.4149055711669669, 0.4149055711669669, 0.4085837625744744, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.39303975900956756, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839, 0.3807888192016839], [0.8807398201726093, 0.8807398201726093, 0.6810498888490331, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3816840444972861, 0.3813134475087317, 0.3813134475087317, 0.3813134475087317, 0.3813134475087317, 0.3523430937726009, 0.3523430937726009, 0.3523430937726009, 0.3523430937726009, 0.3523430937726009, 0.3523430937726009, 0.3523430937726009], [0.8409775825033525, 0.8409775825033525, 0.42547425587961274, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.38116117817006195, 0.35238802717153744, 0.35238802717153744, 0.35238802717153744, 0.35238802717153744, 0.35238802717153744, 0.35238802717153744], [0.9454327875390357, 0.9454327875390357, 0.6387824087002806, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3811408592322302, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864, 0.3508723309067864], [0.887584476596866, 0.887584476596866, 0.6005557175541312, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.3819807211741844, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.38038358718880355, 0.3523614530602802], [0.8604534492425788, 0.8604534492425788, 0.63030696184037, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3811422451492587, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.3783182290441288, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776, 0.35145985487124776], [0.9473172871813721, 0.9473172871813721, 0.5475432226752981, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.38878249152138555, 0.3806145688622473, 0.3806145688622473, 0.3806145688622473, 0.3806145688622473, 0.3806145688622473, 0.3806145688622473, 0.3519384465736292, 0.3519384465736292, 0.3519384465736292, 0.3519384465736292], [0.855343494157695, 0.855343494157695, 0.6582239326959556, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3814339724406091, 0.3805308327896364, 0.3805308327896364, 0.35270227295439394, 0.35270227295439394, 0.35270227295439394, 0.35270227295439394, 0.35270227295439394], [0.9350120009124343, 0.9350120009124343, 0.6464096953886747, 0.4178517646046722, 0.4178517646046722, 0.4178517646046722, 0.4178517646046722, 0.4178517646046722, 0.4142544252490502, 0.4142544252490502, 0.4142544252490502, 0.4142544252490502, 0.4142544252490502, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4094625057828533, 0.4066992688952708, 0.4066992688952708, 0.4066992688952708, 0.4066992688952708, 0.4066992688952708, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.3809834007542931, 0.37775929983755924, 0.37775929983755924, 0.37775929983755924, 0.37775929983755924], [0.9072779867063547, 0.9072779867063547, 0.5608205728226476, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3813062131974789, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303, 0.3790104788229303], [0.91862487068713, 0.91862487068713, 0.6344152322685158, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.38144644567825825, 0.365573053123847, 0.365573053123847], [0.7660839486221795, 0.7660839486221795, 0.4833635162832007, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.38199260080380215, 0.37363724109822677, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736, 0.35189856028683736], [0.8156892192055167, 0.8156892192055167, 0.6019077017328783, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.38119476999041335, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805, 0.3524250251023805], [0.8252579750602671, 0.8252579750602671, 0.588495739927309, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3964099311989311, 0.3521627376365162, 0.3521627376365162, 0.3521627376365162, 0.3521627376365162]]

In [ ]:
print(np.array(cy).shape)

In [ ]:
sleep

In [ ]:
print(vf(vec_x[:,0], vec_x[:,1]))
fig, ax = plt.subplots(figsize = (14,14))
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
#ax.scatter(g_theta1[:, 0].detach(),g_theta1[:, 1].detach(), c="b", alpha=0.8)
ax.plot(g_theta1[:, 0].detach(),g_theta1[:, 1].detach() , 'o', color = 'blue',markersize=15, alpha = 0.2)
ax.plot(vec_x[-1,0], vec_x[-1,1],'v', color = 'red',markersize=15)
ax.plot(0.12, 0.82,'d', color = 'green',markersize=15)
ax.plot(0.54, 0.15,'d', color = 'green',markersize=15)
ax.plot(0.96, 0.15,'d', color = 'green',markersize=15)
ax.set_title('Final TAD configuration', fontsize = 40)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
#ax.legend(['Acquired points', 'Approx. Solution', 'Target'])
plt.savefig('figures/ald_all.pdf')
plt.show()

In [ ]:
vec_x = vec_x.detach()
#v2 = g_theta2_vec.reshape(math.ceil(g_theta2_vec.shape[0]/2), 2)
ii = 0
low = -0.1
high = 1.1
########################
f, ax = plt.subplots(1, 1, figsize=(14, 14))
ax.plot(0.12, 0.82,'d', color = 'green',markersize=15)
ax.plot(0.54, 0.15,'d', color = 'green',markersize=15)
ax.plot(0.96, 0.15,'d', color = 'green',markersize=15)
ax.plot(vec_x[ii,0], vec_x[ii,1],'v', color = 'red',markersize=15)
ax.plot(x_train.detach()[:,0], x_train.detach()[:,1], 's', color = 'black', markersize=15, alpha = 0.2)
#ax.plot(v2.detach()[ii:ii+loc_size+1,0], v2.detach()[ii:ii+loc_size+1,1], 'o', color = 'blue', markersize=15)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Initial Configuration', fontsize = 40)
#ax.legend(['Target', 'Initial Target Candidate', 'Initial 1-sample','Initial 2-sample'])

ax.set_xlim(low, high)
ax.set_ylim(low, high)
plt.savefig('figures/ald_ini.pdf')

In [ ]:
best_A = np.array([[0.858382541775182, 0.858382541775182, 0.7109610530983959, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3816447335461403, 0.3803491580717142, 0.3803491580717142, 0.3803491580717142, 0.35258303363593096, 0.35258303363593096, 0.35258303363593096], [0.7561112149964585, 0.7561112149964585, 0.6446327911156716, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.381125488919018, 0.3510227872399043, 0.3510227872399043, 0.3510227872399043, 0.3510227872399043], [0.8695917296575941, 0.8695917296575941, 0.5546518210185004, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3814566823622931, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174, 0.3801395019462174], [0.6994186606248569, 0.6994186606248569, 0.5935598745776527, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.3812981451492333, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.37880528562092486, 0.3508333094961522, 0.3508333094961522, 0.3508333094961522], [0.7770011552149629, 0.7770011552149629, 0.5897551895608363, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3811522717132394, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.3787270078024409, 0.37374740573489884, 0.35163444325550264, 0.35163444325550264, 0.35163444325550264, 0.35163444325550264, 0.35163444325550264], [0.8079577897205084, 0.8079577897205084, 0.49317231988752175, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.3817514239310018, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.37851198937213093, 0.35151905868571576, 0.35151905868571576, 0.35151905868571576, 0.35151905868571576, 0.35151905868571576], [0.8202767222903737, 0.8202767222903737, 0.5951621950829203, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535, 0.381145001711535], [0.8701160394197105, 0.8701160394197105, 0.6025837779847939, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.39328905670520276, 0.3808184840148786, 0.3808184840148786, 0.3808184840148786, 0.3808184840148786, 0.3808184840148786, 0.3808184840148786, 0.3518185566757966, 0.3518185566757966], [0.7708661160484573, 0.7708661160484573, 0.57047685371561, 0.4706865592885494, 0.4706865592885494, 0.4706865592885494, 0.4706865592885494, 0.4706865592885494, 0.41410711883031176, 0.41410711883031176, 0.41410711883031176, 0.41410711883031176, 0.41410711883031176, 0.4108796933335735, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3784732103820425, 0.3739117032574619, 0.3739117032574619, 0.3739117032574619], [0.8106972383737638, 0.8106972383737638, 0.5444781896854523, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3811386110497042, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.3779476911870878, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664, 0.35195956801927664]])


In [ ]:
torch.arange(0, 830, 21)

In [ ]:
 (1 + 2*(nrep+1) + 2*(nrep-1)) * (ninit+1)